# Historical Modeling

Tujuan: Mengevaluasi pengaruh *historical features* terhadap performa model prediksi hasil pertandingan sepak bola internasional serta membandingkannya dengan *baseline model*.

## 1. Load Historical Dataset

Memuat dataset hasil *Historical Feature Engineering* yang akan digunakan untuk membangun dan mengevaluasi model.

In [27]:
import pandas as pd

In [28]:
historical_df = pd.read_csv("../data/processed/historical_features.csv", parse_dates=["date"])

In [29]:
historical_df.head()

,date,home_team,away_team,tournament,neutral,home_last5_winrate,away_last5_winrate,home_last10_winrate,away_last10_winrate,home_avg_goals_scored,away_avg_goals_scored,home_avg_goals_conceded,away_avg_goals_conceded,home_goal_difference_form,away_goal_difference_form,match_result
0,1872-11-30,Scotland,England,Friendly,False,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,D
1,1873-03-08,England,Scotland,Friendly,False,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,H
2,1874-03-07,Scotland,England,Friendly,False,0.000000,0.500000,0.000000,0.500000,1.000000,2.000000,2.000000,1.000000,-1.000000,1.000000,H
3,1875-03-06,England,Scotland,Friendly,False,0.333333,0.333333,0.333333,0.333333,1.666667,1.333333,1.333333,1.666667,0.333333,-0.333333,D
4,1876-03-04,Scotland,England,Friendly,False,0.250000,0.250000,0.250000,0.250000,1.500000,1.750000,1.750000,1.500000,-0.250000,0.250000,H


In [30]:
historical_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 49433 entries, 0 to 49432
Data columns (total 16 columns):
 #   Column                     Non-Null Count  Dtype         
---  ------                     --------------  -----         
 0   date                       49433 non-null  datetime64[us]
 1   home_team                  49433 non-null  str           
 2   away_team                  49433 non-null  str           
 3   tournament                 49433 non-null  str           
 4   neutral                    49433 non-null  bool          
 5   home_last5_winrate         49292 non-null  float64       
 6   away_last5_winrate         49238 non-null  float64       
 7   home_last10_winrate        49292 non-null  float64       
 8   away_last10_winrate        49238 non-null  float64       
 9   home_avg_goals_scored      49292 non-null  float64       
 10  away_avg_goals_scored      49238 non-null  float64       
 11  home_avg_goals_conceded    49292 non-null  float64       
 12  away_avg_goals_

In [31]:
historical_df.shape

(49433, 16)

Dataset berhasil dimuat dan berisi seluruh fitur baseline serta sepuluh *historical features* yang telah dibangun pada notebook sebelumnya. Seluruh fitur tersebut merepresentasikan kondisi masing-masing tim **sebelum pertandingan berlangsung**, sehingga dapat digunakan sebagai masukan model tanpa menimbulkan *future data leakage*.

Pada tahap berikutnya, fitur (*X*) dan target (*y*) akan dipisahkan sebagai persiapan proses pelatihan model.

## 2. Feature & Target Separation
Sebelum membangun model, dataset dipisahkan menjadi **fitur (X)** dan **target (y)**. Variabel target yang akan diprediksi adalah `match_result`, sedangkan seluruh kolom lainnya yang relevan digunakan sebagai fitur prediktor. Kolom `date` tidak digunakan dalam proses pelatihan model karena hanya berfungsi sebagai acuan untuk melakukan *time-based train-test split*.

In [32]:
feature_columns = [
    "home_team",
    "away_team",
    "tournament",
    "neutral",

    "home_last5_winrate",
    "away_last5_winrate",

    "home_last10_winrate",
    "away_last10_winrate",

    "home_avg_goals_scored",
    "away_avg_goals_scored",

    "home_avg_goals_conceded",
    "away_avg_goals_conceded",

    "home_goal_difference_form",
    "away_goal_difference_form"
]

X = historical_df[feature_columns]
y = historical_df["match_result"]

In [33]:
# VERIFIKASI
print("Feature Shape :", X.shape)
print("Target Shape  :", y.shape)

Feature Shape : (49433, 14)
Target Shape  : (49433,)


Pemisahan fitur dan target berhasil dilakukan. Model akan menggunakan **14 fitur**, yang terdiri dari:
- 4 fitur dasar (*baseline features*),
- 10 fitur historis (*historical features*).
Variabel target tetap berupa `match_result` dengan tiga kelas (`H`, `D`, dan `A`). Sementara itu, kolom `date` sengaja tidak disertakan sebagai fitur karena hanya digunakan untuk menjaga urutan kronologis saat proses *train-test split*.

## 3. Time-Based Train-Test Split

Dataset dibagi menjadi data latih (*training set*) dan data uji (*test set*) berdasarkan urutan waktu (*time-based split*). Berbeda dengan *random split*, pendekatan ini mempertahankan urutan kronologis pertandingan sehingga model hanya belajar dari pertandingan yang terjadi di masa lalu untuk memprediksi pertandingan di masa depan. Metode ini lebih mencerminkan kondisi prediksi di dunia nyata (*real-world prediction*) serta menghindari *future data leakage*. Untuk menjaga konsistensi eksperimen, proporsi pembagian data tetap sama seperti pada *Baseline Modeling*, yaitu **80% data latih** dan **20% data uji**.

In [34]:
split_index = int(len(historical_df) * 0.8)

X_train = X.iloc[:split_index]
X_test = X.iloc[split_index:]

y_train = y.iloc[:split_index]
y_test = y.iloc[split_index:]

In [35]:
# VALIDASI
print(f"Training samples : {len(X_train):,}")
print(f"Testing samples  : {len(X_test):,}")

Training samples : 39,546
Testing samples  : 9,887


In [36]:
print("Training period")
print(
    historical_df.iloc[0]["date"],
    "---",
    historical_df.iloc[split_index - 1]["date"]
)

print()

print("Testing period")
print(
    historical_df.iloc[split_index]["date"],
    "---",
    historical_df.iloc[-1]["date"]
)

Training period
1872-11-30 00:00:00 --- 2016-03-25 00:00:00

Testing period
2016-03-25 00:00:00 --- 2026-06-18 00:00:00


Dataset berhasil dibagi menjadi data latih dan data uji menggunakan pendekatan *time-based split*. Seluruh pertandingan pada data latih terjadi lebih awal dibandingkan data uji, sehingga model hanya memanfaatkan informasi historis yang tersedia sebelum periode evaluasi. Dengan mempertahankan strategi pembagian data yang sama seperti pada *Baseline Modeling*, hasil evaluasi pada notebook ini dapat dibandingkan secara langsung untuk mengukur pengaruh penambahan *historical features* terhadap performa model.

## 4. Historical Modeling

Pada tahap ini akan dibangun dua model klasifikasi menggunakan dataset yang telah diperkaya dengan *historical features*. Proses pemodelan tetap mengikuti pendekatan yang digunakan pada *Baseline Modeling* agar hasil evaluasi dapat dibandingkan secara adil. Perbedaannya hanya terletak pada informasi yang diberikan kepada model, yaitu penambahan fitur historis yang merepresentasikan performa masing-masing tim sebelum pertandingan berlangsung.

### 4.1 Feature Types
Dataset terdiri atas fitur kategorikal dan numerik sehingga masing-masing memerlukan perlakuan yang berbeda pada tahap *preprocessing*. Fitur kategorikal akan diubah menjadi representasi numerik menggunakan **One-Hot Encoding**, sedangkan fitur numerik akan diteruskan (*passthrough*) tanpa transformasi tambahan.

In [37]:
# KATEGORIKAL
categorical_features = [
    "home_team",
    "away_team",
    "tournament"
]

# NUMERIK
numerical_features = [
    "neutral",

    "home_last5_winrate",
    "away_last5_winrate",

    "home_last10_winrate",
    "away_last10_winrate",

    "home_avg_goals_scored",
    "away_avg_goals_scored",

    "home_avg_goals_conceded",
    "away_avg_goals_conceded",

    "home_goal_difference_form",
    "away_goal_difference_form"
]

# VALIDASI
print("Categorical Features :", len(categorical_features))
print("Numerical Features   :", len(numerical_features))
print("Total Features       :", len(categorical_features) + len(numerical_features))

Categorical Features : 3
Numerical Features   : 11
Total Features       : 14


Seluruh fitur berhasil dikelompokkan berdasarkan tipe datanya. Terdapat **3 fitur kategorikal** (`home_team`, `away_team`, dan `tournament`) yang akan diproses menggunakan *One-Hot Encoding*. Sementara itu, **11 fitur numerik** terdiri atas satu fitur boolean (`neutral`) dan sepuluh *historical features* yang akan diteruskan langsung ke model tanpa transformasi tambahan. Pemisahan ini memungkinkan seluruh proses *preprocessing* dilakukan secara otomatis melalui *Pipeline* pada tahap berikutnya.

### 4.2 Modeling Pipeline
Seluruh proses *preprocessing* dan pelatihan model dilakukan menggunakan **Pipeline**. Pendekatan ini memastikan bahwa proses transformasi hanya dipelajari dari data latih (*training data*) sehingga terhindar dari *data leakage*. Selain itu, penggunaan *Pipeline* membuat proses pemodelan menjadi lebih modular, konsisten, dan mudah digunakan kembali pada eksperimen berikutnya.

In [38]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer

# karena logistic regression tidak bisa menangani NaN, maka kita perlu mengisi missing value dengan 0
numeric_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="constant", fill_value=0))
    ]
)


preprocessor = ColumnTransformer(
    transformers=[
        (
            "categorical",
            OneHotEncoder(handle_unknown="ignore"),
            categorical_features
        ),
        (
            "numerical",
            numeric_transformer,
            numerical_features
        )
    ]
)

### 4.3 Logistic Regression
Logistic Regression kembali digunakan sebagai model baseline utama karena memberikan performa terbaik pada eksperimen sebelumnya. Dengan mempertahankan algoritma yang sama, setiap perubahan performa pada notebook ini dapat diatribusikan pada penambahan *historical features*, bukan karena pergantian model.

In [39]:
from sklearn.linear_model import LogisticRegression

logistic_pipeline = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        (
            "classifier",
            LogisticRegression(
                random_state=42,
                max_iter=1000
            )
        )
    ]
)

In [40]:
logistic_pipeline.fit(X_train, y_train)
y_pred_lr = logistic_pipeline.predict(X_test)

Model Logistic Regression berhasil dilatih menggunakan kombinasi fitur dasar dan *historical features*. Selanjutnya, hasil prediksi akan dievaluasi untuk melihat apakah penambahan informasi historis mampu meningkatkan performa dibandingkan dengan *Baseline Modeling*.

### 4.4 Random Forest
Random Forest digunakan sebagai model pembanding (*benchmark*) untuk melihat bagaimana algoritma berbasis pohon keputusan memanfaatkan *historical features* yang telah ditambahkan.

In [41]:
from sklearn.ensemble import RandomForestClassifier

rf_pipeline = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        (
            "classifier",
            RandomForestClassifier(
                random_state=42
            )
        )
    ]
)

In [42]:
rf_pipeline.fit(X_train, y_train)
y_pred_rf = rf_pipeline.predict(X_test)

#### Insight
Model Random Forest juga berhasil dilatih menggunakan dataset yang telah diperkaya dengan *historical features*. Pada tahap evaluasi berikutnya, performanya akan dibandingkan dengan Logistic Regression serta hasil *Baseline Modeling* untuk mengetahui seberapa besar pengaruh fitur historis terhadap masing-masing algoritma.

## 5. Model Evaluation
Setelah model selesai dilatih, langkah berikutnya adalah mengevaluasi performanya menggunakan data uji (*test set*).
Evaluasi dilakukan menggunakan beberapa metrik klasifikasi, yaitu:
- Accuracy
- Precision
- Recall
- F1 Score
Selain itu, *Classification Report* dan *Confusion Matrix* juga digunakan untuk memahami perilaku model pada masing-masing kelas hasil pertandingan (`H`, `D`, dan `A`).

In [43]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix
)

### 5.1 Logistic Regression Evaluation

In [44]:
lr_accuracy = accuracy_score(y_test, y_pred_lr)

lr_precision = precision_score(
    y_test,
    y_pred_lr,
    average="weighted"
)

lr_recall = recall_score(
    y_test,
    y_pred_lr,
    average="weighted"
)

lr_f1 = f1_score(
    y_test,
    y_pred_lr,
    average="weighted"
)


print("=== Logistic Regression ===")
print(f"Accuracy  : {lr_accuracy:.4f}")
print(f"Precision : {lr_precision:.4f}")
print(f"Recall    : {lr_recall:.4f}")
print(f"F1 Score  : {lr_f1:.4f}")



=== Logistic Regression ===
Accuracy  : 0.5805
Precision : 0.5268
Recall    : 0.5805
F1 Score  : 0.5209


In [45]:
print(classification_report(y_test, y_pred_lr))

              precision    recall  f1-score   support

           A       0.57      0.54      0.55      2868
           D       0.32      0.06      0.10      2305
           H       0.60      0.86      0.71      4714

    accuracy                           0.58      9887
   macro avg       0.50      0.49      0.45      9887
weighted avg       0.53      0.58      0.52      9887



In [46]:
cm_lr = confusion_matrix(
    y_test,
    y_pred_lr
)

cm_lr

array([[1547,  154, 1167],
       [ 646,  133, 1526],
       [ 531,  124, 4059]])


#### Insight: Logistic Regression
Logistic Regression berhasil memanfaatkan *historical features* untuk meningkatkan performa dibandingkan *Baseline Modeling*. Accuracy meningkat dari **57,30%** menjadi **58,05%**, disertai peningkatan kecil pada Precision, Recall, dan F1 Score.

Berdasarkan *classification report*, model masih menunjukkan performa terbaik pada kelas **Home Win (H)** dengan nilai *recall* mencapai **86%**, yang menunjukkan sebagian besar kemenangan tim kandang berhasil dikenali dengan baik. Sementara itu, kelas **Draw (D)** masih menjadi tantangan utama. Meskipun terdapat sedikit peningkatan dibandingkan model baseline, nilai *recall* masih sangat rendah (**6%**), yang menunjukkan bahwa pertandingan imbang masih sulit dibedakan hanya menggunakan kombinasi fitur dasar dan *historical features*.

Secara keseluruhan, hasil ini menunjukkan bahwa penambahan informasi historis berhasil meningkatkan kemampuan prediksi model, namun belum cukup untuk sepenuhnya menangkap kondisi pertandingan yang berakhir imbang.

### 5.2 Random Forest Evaluation

In [47]:
rf_accuracy = accuracy_score(y_test, y_pred_rf)

rf_precision = precision_score(
    y_test,
    y_pred_rf,
    average="weighted"
)

rf_recall = recall_score(
    y_test,
    y_pred_rf,
    average="weighted"
)

rf_f1 = f1_score(
    y_test,
    y_pred_rf,
    average="weighted"
)

print("=== Random Forest ===")
print(f"Accuracy  : {rf_accuracy:.4f}")
print(f"Precision : {rf_precision:.4f}")
print(f"Recall    : {rf_recall:.4f}")
print(f"F1 Score  : {rf_f1:.4f}")

=== Random Forest ===
Accuracy  : 0.5467
Precision : 0.4955
Recall    : 0.5467
F1 Score  : 0.4717


In [48]:
print(classification_report(y_test, y_pred_rf))

              precision    recall  f1-score   support

           A       0.56      0.40      0.47      2868
           D       0.31      0.03      0.05      2305
           H       0.55      0.89      0.68      4714

    accuracy                           0.55      9887
   macro avg       0.47      0.44      0.40      9887
weighted avg       0.50      0.55      0.47      9887



In [49]:
cm_rf = confusion_matrix(
    y_test,
    y_pred_rf
)

cm_rf

array([[1155,   68, 1645],
       [ 470,   65, 1770],
       [ 450,   79, 4185]])

#### Insight: Random Forest

Random Forest juga mengalami peningkatan performa setelah menggunakan *historical features*. Accuracy meningkat dari **50,84%** pada *Baseline Modeling* menjadi **54,67%**, disertai peningkatan pada seluruh metrik evaluasi. Peningkatan ini menunjukkan bahwa algoritma berbasis pohon keputusan mampu memanfaatkan informasi historis seperti *win rate*, rata-rata gol, dan performa tim untuk menghasilkan keputusan yang lebih baik dibandingkan ketika hanya menggunakan fitur dasar.

Meskipun demikian, model masih mengalami kesulitan dalam memprediksi kelas **Draw (D)**. Nilai *recall* hanya mencapai **3%**, yang mengindikasikan bahwa sebagian besar pertandingan imbang masih diprediksi sebagai kemenangan salah satu tim. Hasil ini menunjukkan bahwa *historical features* memberikan dampak yang cukup signifikan terhadap Random Forest, namun masih belum mampu mengatasi kompleksitas pola pertandingan yang berakhir imbang.

### 5.3 Historical Model Comparison


In [52]:
comparison_df = pd.DataFrame(
    {
        "Model": [
            "Logistic Regression",
            "Random Forest"
        ],
        "Accuracy": [
            lr_accuracy,
            rf_accuracy
        ],
        "Precision": [
            lr_precision,
            rf_precision
        ],
        "Recall": [
            lr_recall,
            rf_recall
        ],
        "F1 Score": [
            lr_f1,
            rf_f1
        ]
    }
)

In [51]:
comparison_df.sort_values(
    by="Accuracy",
    ascending=False
)

,Model,Accuracy,Precision,Recall,F1 Score
0,Logistic Regression,0.580459,0.526805,0.580459,0.520899
1,Random Forest,0.546677,0.495492,0.546677,0.471682


#### Insight
Berdasarkan hasil evaluasi, **Logistic Regression** kembali menjadi model dengan performa terbaik pada eksperimen ini dengan Accuracy sebesar **58,05%**, mengungguli **Random Forest** yang memperoleh Accuracy sebesar **54,67%**. Dibandingkan dengan *Baseline Modeling*, kedua model sama-sama mengalami peningkatan performa setelah ditambahkan *historical features*. Logistic Regression meningkat sekitar **0,75 poin persentase**, sedangkan Random Forest mengalami peningkatan yang lebih besar, yaitu sekitar **3,83 poin persentase**.

Temuan ini menunjukkan bahwa informasi historis mengenai performa tim memang memberikan sinyal prediktif yang berguna bagi kedua algoritma. Meskipun peningkatan pada Logistic Regression relatif kecil, konsistensi kenaikan pada seluruh metrik evaluasi mengindikasikan bahwa *historical features* berhasil memperkaya informasi yang tersedia sebelum pertandingan berlangsung. Namun demikian, kedua model masih memiliki kelemahan yang sama, yaitu kesulitan dalam mengenali pertandingan yang berakhir **Draw (D)**. Hal ini mengindikasikan bahwa performa historis saja belum cukup untuk merepresentasikan keseimbangan kekuatan dua tim secara menyeluruh.

Oleh karena itu, hasil eksperimen ini menjadi justifikasi yang kuat untuk melanjutkan pengembangan fitur yang lebih representatif, seperti **Elo Rating**, yang dirancang untuk menggambarkan kekuatan relatif antar tim secara lebih dinamis.